# LlamaIndex

**Module:** 09-llm-frameworks

**Notebook:** `03-llamaindex.ipynb`

This expanded lesson goes beyond definitions: each topic includes *why it matters*, *how it works*, intuition, pitfalls, and when to use it—plus runnable Python demos, comparison aids, and exercises.


## Learning Objectives

By the end of this notebook, you will be able to:

- Explain and apply **Core Concepts** with clear contracts and failure modes
- Explain and apply **Data Connectors** with clear contracts and failure modes
- Explain and apply **Indexing & Nodes** with clear contracts and failure modes
- Explain and apply **Retrieval** with clear contracts and failure modes
- Explain and apply **RAG Pipeline** with clear contracts and failure modes
- Explain and apply **Query vs Chat Engines** with clear contracts and failure modes
- Evaluate tradeoffs (quality, cost, latency, safety) for designs in this lesson
- Implement small Python prototypes that make the ideas testable


## How to Use This Notebook

1. Read the topic sections fully—do not jump only to code.
2. Run each demo; then change inputs to break them and fix them.
3. API examples use placeholders like `YOUR_API_KEY` or `os.environ.get(...)`.
4. Keep secrets out of git; treat prompts/tool schemas as versioned code.
5. Complete the **Try It Yourself** exercises before moving on.


### Pipeline walkthrough — LlamaIndex

```mermaid
flowchart LR
  A[Problem / user goal] --> B[Contract: IO + constraints]
  B --> C[Implement core path]
  C --> D[Validate / guardrails]
  D --> E[Eval fixtures]
  E --> F[Observe in production]
  F -->|regressions| B
```

```text
goal -> contract -> implement -> validate -> evaluate -> monitor -> revise
```


## Curriculum Map

This notebook's spine (preserve/cover all of these):

1. **Core Concepts**
2. **Data Connectors**
3. **Indexing & Nodes**
4. **Retrieval**
5. **RAG Pipeline**
6. **Query vs Chat Engines**

Read top-to-bottom once, then revisit weak spots with the exercises.


## Core Concepts

### Definition
**Core Concepts** is a core building block in 03-llamaindex within LLM frameworks. Treat it as an orchestration layer—useful only when it clarifies ownership of steps: something you can name, version, test, and operate.

### Why it matters
In LLM frameworks, weak designs around Core Concepts typically surface as framework lock-in, opaque magic, and untested compositions. Investing here improves reliability, debuggability, and the ability to change models later.

### How it works
For Core Concepts: (1) write an explicit input/output contract, (2) implement the minimal happy path, (3) validate and add guardrails, (4) cover golden + adversarial fixtures, (5) wire observability. Your durable artifacts should look like runnable chains, indexes, and portable pipelines.

### Intuition
Explain Core Concepts as an orchestration layer—useful only when it clarifies ownership of steps. If a new engineer cannot tell what is trusted input, what is allowed action, and what 'done' means, the design is still fuzzy.

### Pitfalls
- Treating Core Concepts as a one-time playground experiment instead of a versioned artifact
- No success criteria or eval set for Core Concepts
- Ignoring cost/latency tradeoffs while chasing marginal quality
- Missing adversarial cases typical of LLM frameworks: framework lock-in, opaque magic, and untested compositions

### When to use
Use Core Concepts when your product path depends on this concern in LLM frameworks. Prefer the simplest design that meets quality, latency, and safety budgets—and prove it with fixtures.

### Quick reference

| Lens | Question |
|------|----------|
| Product | What user outcome does Core Concepts improve? |
| Engineering | What is the interface / data contract? |
| Safety | What can go wrong if it fails open? |
| Ops | How will we notice regressions? |


In [ ]:
# Demo: make "Core Concepts" concrete as a checkable contract
from dataclasses import dataclass, field, asdict
import json

@dataclass
class ConceptContract:
    name: str = "Core Concepts"
    notebook: str = "03-llamaindex"
    must_have: list = field(default_factory=lambda: [
        "clear inputs/outputs",
        "failure behavior defined",
        "eval fixtures exist",
    ])
    risks: list = field(default_factory=lambda: [
        "silent quality drift",
        "unbounded cost/latency",
    ])

    def health(self) -> dict:
        return {
            "concept": self.name,
            "checks": len(self.must_have),
            "risks": len(self.risks),
            "ready_for_design_review": len(self.must_have) >= 3,
        }

contract_0 = ConceptContract()
print(json.dumps({"contract": asdict(contract_0), "health": contract_0.health()}, indent=2))


In [ ]:
# Demo: before/after quality rubric for "Core Concepts"
def score_artifact(artifact: dict, rubric: list[str]) -> dict:
    missing = [r for r in rubric if not artifact.get(r)]
    return {"score": round(1 - len(missing)/max(1,len(rubric)), 2), "missing": missing}

rubric = ["definition", "example", "failure_mode", "metric"]
weak = {"definition": "Core Concepts"}
strong = {"definition": "Core Concepts", "example": "worked example", "failure_mode": "empty input", "metric": "exact_match"}
print("weak", score_artifact(weak, rubric))
print("strong", score_artifact(strong, rubric))


In [ ]:
# Demo: operational checklist runner for "Core Concepts"
checks = {
    "has_owner": True,
    "has_eval_set": True,
    "has_token_budget": False,
    "has_alert": False,
}
failed = [k for k, ok in checks.items() if not ok]
print({"topic": "Core Concepts", "passed": len(checks)-len(failed), "failed": failed})


In [ ]:
# Demo: decision table for applying "Core Concepts"
options = [
    {"option": "baseline_simple", "quality": 0.7, "cost": 1, "ops": 0.9},
    {"option": "advanced_core_concept", "quality": 0.85, "cost": 3, "ops": 0.6},
]
for o in options:
    o["utility"] = round(o["quality"] * 2 - 0.3*o["cost"] + 0.5*o["ops"], 3)
best = max(options, key=lambda x: x["utility"])
print("ranked:", sorted(options, key=lambda x: -x["utility"]))
print("prefer:", best["option"])


## Data Connectors

### Definition
**Data Connectors** is a core building block in 03-llamaindex within LLM frameworks. Treat it as an orchestration layer—useful only when it clarifies ownership of steps: something you can name, version, test, and operate.

### Why it matters
In LLM frameworks, weak designs around Data Connectors typically surface as framework lock-in, opaque magic, and untested compositions. Investing here improves reliability, debuggability, and the ability to change models later.

### How it works
For Data Connectors: (1) write an explicit input/output contract, (2) implement the minimal happy path, (3) validate and add guardrails, (4) cover golden + adversarial fixtures, (5) wire observability. Your durable artifacts should look like runnable chains, indexes, and portable pipelines.

### Intuition
Explain Data Connectors as an orchestration layer—useful only when it clarifies ownership of steps. If a new engineer cannot tell what is trusted input, what is allowed action, and what 'done' means, the design is still fuzzy.

### Pitfalls
- Treating Data Connectors as a one-time playground experiment instead of a versioned artifact
- No success criteria or eval set for Data Connectors
- Ignoring cost/latency tradeoffs while chasing marginal quality
- Missing adversarial cases typical of LLM frameworks: framework lock-in, opaque magic, and untested compositions

### When to use
Use Data Connectors when your product path depends on this concern in LLM frameworks. Prefer the simplest design that meets quality, latency, and safety budgets—and prove it with fixtures.


In [ ]:
# Demo: make "Data Connectors" concrete as a checkable contract
from dataclasses import dataclass, field, asdict
import json

@dataclass
class ConceptContract:
    name: str = "Data Connectors"
    notebook: str = "03-llamaindex"
    must_have: list = field(default_factory=lambda: [
        "clear inputs/outputs",
        "failure behavior defined",
        "eval fixtures exist",
    ])
    risks: list = field(default_factory=lambda: [
        "silent quality drift",
        "unbounded cost/latency",
    ])

    def health(self) -> dict:
        return {
            "concept": self.name,
            "checks": len(self.must_have),
            "risks": len(self.risks),
            "ready_for_design_review": len(self.must_have) >= 3,
        }

contract_1 = ConceptContract()
print(json.dumps({"contract": asdict(contract_1), "health": contract_1.health()}, indent=2))


In [ ]:
# Demo: before/after quality rubric for "Data Connectors"
def score_artifact(artifact: dict, rubric: list[str]) -> dict:
    missing = [r for r in rubric if not artifact.get(r)]
    return {"score": round(1 - len(missing)/max(1,len(rubric)), 2), "missing": missing}

rubric = ["definition", "example", "failure_mode", "metric"]
weak = {"definition": "Data Connectors"}
strong = {"definition": "Data Connectors", "example": "worked example", "failure_mode": "empty input", "metric": "exact_match"}
print("weak", score_artifact(weak, rubric))
print("strong", score_artifact(strong, rubric))


In [ ]:
# Demo: operational checklist runner for "Data Connectors"
checks = {
    "has_owner": True,
    "has_eval_set": True,
    "has_token_budget": False,
    "has_alert": False,
}
failed = [k for k, ok in checks.items() if not ok]
print({"topic": "Data Connectors", "passed": len(checks)-len(failed), "failed": failed})


### Worked scenario — Data Connectors

**Situation:** A team wants to productionize a feature involving **Data Connectors**.

**Walkthrough:**
1. Write a one-sentence success metric.
2. Define inputs, outputs, and hard constraints.
3. Implement the smallest demo that can fail loudly.
4. Add one adversarial fixture (empty, hostile, or oversized input).
5. Decide ship/no-ship using the metric—not eloquence.


## Indexing & Nodes

### Definition
**Indexing & Nodes** is a core building block in 03-llamaindex within LLM frameworks. Treat it as an orchestration layer—useful only when it clarifies ownership of steps: something you can name, version, test, and operate.

### Why it matters
In LLM frameworks, weak designs around Indexing & Nodes typically surface as framework lock-in, opaque magic, and untested compositions. Investing here improves reliability, debuggability, and the ability to change models later.

### How it works
For Indexing & Nodes: (1) write an explicit input/output contract, (2) implement the minimal happy path, (3) validate and add guardrails, (4) cover golden + adversarial fixtures, (5) wire observability. Your durable artifacts should look like runnable chains, indexes, and portable pipelines.

### Intuition
Explain Indexing & Nodes as an orchestration layer—useful only when it clarifies ownership of steps. If a new engineer cannot tell what is trusted input, what is allowed action, and what 'done' means, the design is still fuzzy.

### Pitfalls
- Treating Indexing & Nodes as a one-time playground experiment instead of a versioned artifact
- No success criteria or eval set for Indexing & Nodes
- Ignoring cost/latency tradeoffs while chasing marginal quality
- Missing adversarial cases typical of LLM frameworks: framework lock-in, opaque magic, and untested compositions

### When to use
Use Indexing & Nodes when your product path depends on this concern in LLM frameworks. Prefer the simplest design that meets quality, latency, and safety budgets—and prove it with fixtures.


In [ ]:
# Demo: make "Indexing & Nodes" concrete as a checkable contract
from dataclasses import dataclass, field, asdict
import json

@dataclass
class ConceptContract:
    name: str = "Indexing & Nodes"
    notebook: str = "03-llamaindex"
    must_have: list = field(default_factory=lambda: [
        "clear inputs/outputs",
        "failure behavior defined",
        "eval fixtures exist",
    ])
    risks: list = field(default_factory=lambda: [
        "silent quality drift",
        "unbounded cost/latency",
    ])

    def health(self) -> dict:
        return {
            "concept": self.name,
            "checks": len(self.must_have),
            "risks": len(self.risks),
            "ready_for_design_review": len(self.must_have) >= 3,
        }

contract_2 = ConceptContract()
print(json.dumps({"contract": asdict(contract_2), "health": contract_2.health()}, indent=2))


In [ ]:
# Demo: before/after quality rubric for "Indexing & Nodes"
def score_artifact(artifact: dict, rubric: list[str]) -> dict:
    missing = [r for r in rubric if not artifact.get(r)]
    return {"score": round(1 - len(missing)/max(1,len(rubric)), 2), "missing": missing}

rubric = ["definition", "example", "failure_mode", "metric"]
weak = {"definition": "Indexing & Nodes"}
strong = {"definition": "Indexing & Nodes", "example": "worked example", "failure_mode": "empty input", "metric": "exact_match"}
print("weak", score_artifact(weak, rubric))
print("strong", score_artifact(strong, rubric))


In [ ]:
# Demo: operational checklist runner for "Indexing & Nodes"
checks = {
    "has_owner": True,
    "has_eval_set": True,
    "has_token_budget": False,
    "has_alert": False,
}
failed = [k for k, ok in checks.items() if not ok]
print({"topic": "Indexing & Nodes", "passed": len(checks)-len(failed), "failed": failed})


## Retrieval

### Definition
**Retrieval** is a core building block in 03-llamaindex within LLM frameworks. Treat it as an orchestration layer—useful only when it clarifies ownership of steps: something you can name, version, test, and operate.

### Why it matters
In LLM frameworks, weak designs around Retrieval typically surface as framework lock-in, opaque magic, and untested compositions. Investing here improves reliability, debuggability, and the ability to change models later.

### How it works
For Retrieval: (1) write an explicit input/output contract, (2) implement the minimal happy path, (3) validate and add guardrails, (4) cover golden + adversarial fixtures, (5) wire observability. Your durable artifacts should look like runnable chains, indexes, and portable pipelines.

### Intuition
Explain Retrieval as an orchestration layer—useful only when it clarifies ownership of steps. If a new engineer cannot tell what is trusted input, what is allowed action, and what 'done' means, the design is still fuzzy.

### Pitfalls
- Treating Retrieval as a one-time playground experiment instead of a versioned artifact
- No success criteria or eval set for Retrieval
- Ignoring cost/latency tradeoffs while chasing marginal quality
- Missing adversarial cases typical of LLM frameworks: framework lock-in, opaque magic, and untested compositions

### When to use
Use Retrieval when your product path depends on this concern in LLM frameworks. Prefer the simplest design that meets quality, latency, and safety budgets—and prove it with fixtures.


In [ ]:
# Demo: make "Retrieval" concrete as a checkable contract
from dataclasses import dataclass, field, asdict
import json

@dataclass
class ConceptContract:
    name: str = "Retrieval"
    notebook: str = "03-llamaindex"
    must_have: list = field(default_factory=lambda: [
        "clear inputs/outputs",
        "failure behavior defined",
        "eval fixtures exist",
    ])
    risks: list = field(default_factory=lambda: [
        "silent quality drift",
        "unbounded cost/latency",
    ])

    def health(self) -> dict:
        return {
            "concept": self.name,
            "checks": len(self.must_have),
            "risks": len(self.risks),
            "ready_for_design_review": len(self.must_have) >= 3,
        }

contract_3 = ConceptContract()
print(json.dumps({"contract": asdict(contract_3), "health": contract_3.health()}, indent=2))


In [ ]:
GOLDEN = [{"x": "charged twice", "y": "billing"}, {"x": "SSO", "y": "auth"}]

def predict(x: str) -> str:
    return "billing" if "charge" in x.lower() or "invoice" in x.lower() else ("auth" if "sso" in x.lower() or "login" in x.lower() else "other")

def eval_prompt(name: str):
    rows = [(g["x"], g["y"], predict(g["x"])) for g in GOLDEN]
    acc = sum(y == p for _, y, p in rows) / len(rows)
    return {"name": name, "accuracy": acc, "rows": rows}

print(eval_prompt("v1"))


In [ ]:
import hashlib

def prompt_version(text: str) -> str:
    return "pv_" + hashlib.sha1(text.encode()).hexdigest()[:10]

a = "Label tickets carefully"
b = "Label tickets as billing|auth|outage|other. JSON only."
print(prompt_version(a), prompt_version(b))


### Worked scenario — Retrieval

**Situation:** A team wants to productionize a feature involving **Retrieval**.

**Walkthrough:**
1. Write a one-sentence success metric.
2. Define inputs, outputs, and hard constraints.
3. Implement the smallest demo that can fail loudly.
4. Add one adversarial fixture (empty, hostile, or oversized input).
5. Decide ship/no-ship using the metric—not eloquence.


## RAG Pipeline

### Definition
**RAG Pipeline** is a core building block in 03-llamaindex within LLM frameworks. Treat it as an orchestration layer—useful only when it clarifies ownership of steps: something you can name, version, test, and operate.

### Why it matters
In LLM frameworks, weak designs around RAG Pipeline typically surface as framework lock-in, opaque magic, and untested compositions. Investing here improves reliability, debuggability, and the ability to change models later.

### How it works
For RAG Pipeline: (1) write an explicit input/output contract, (2) implement the minimal happy path, (3) validate and add guardrails, (4) cover golden + adversarial fixtures, (5) wire observability. Your durable artifacts should look like runnable chains, indexes, and portable pipelines.

### Intuition
Explain RAG Pipeline as an orchestration layer—useful only when it clarifies ownership of steps. If a new engineer cannot tell what is trusted input, what is allowed action, and what 'done' means, the design is still fuzzy.

### Pitfalls
- Treating RAG Pipeline as a one-time playground experiment instead of a versioned artifact
- No success criteria or eval set for RAG Pipeline
- Ignoring cost/latency tradeoffs while chasing marginal quality
- Missing adversarial cases typical of LLM frameworks: framework lock-in, opaque magic, and untested compositions

### When to use
Use RAG Pipeline when your product path depends on this concern in LLM frameworks. Prefer the simplest design that meets quality, latency, and safety budgets—and prove it with fixtures.


In [ ]:
# Demo: make "RAG Pipeline" concrete as a checkable contract
from dataclasses import dataclass, field, asdict
import json

@dataclass
class ConceptContract:
    name: str = "RAG Pipeline"
    notebook: str = "03-llamaindex"
    must_have: list = field(default_factory=lambda: [
        "clear inputs/outputs",
        "failure behavior defined",
        "eval fixtures exist",
    ])
    risks: list = field(default_factory=lambda: [
        "silent quality drift",
        "unbounded cost/latency",
    ])

    def health(self) -> dict:
        return {
            "concept": self.name,
            "checks": len(self.must_have),
            "risks": len(self.risks),
            "ready_for_design_review": len(self.must_have) >= 3,
        }

contract_4 = ConceptContract()
print(json.dumps({"contract": asdict(contract_4), "health": contract_4.health()}, indent=2))


In [ ]:
class Pipeline:
    def __init__(self):
        self.stages = []
    def add(self, name, fn):
        self.stages.append((name, fn)); return self
    def run(self, data):
        audit = []
        for name, fn in self.stages:
            data = fn(data)
            audit.append({"stage": name, "type": type(data).__name__})
        return data, audit

result, audit = (
    Pipeline()
    .add("normalize", lambda s: s.strip().lower())
    .add("tokens", lambda s: s.split())
    .add("features", lambda toks: {"n": len(toks), "head": toks[:3]})
    .run("  SSO Login Loop  ")
)
print(result); print(audit)


In [ ]:
# Parallel fan-out / fan-in sketch
from concurrent.futures import ThreadPoolExecutor

def analyze(kind, text):
    return {"kind": kind, "len": len(text)}

text = "investigate checkout latency"
with ThreadPoolExecutor(max_workers=3) as ex:
    parts = list(ex.map(lambda k: analyze(k, text), ["security", "perf", "ux"]))
merged = {p["kind"]: p["len"] for p in parts}
print(merged)


## Query vs Chat Engines

### Definition
**Query vs Chat Engines** helps you choose among alternatives using explicit criteria rather than hype.

### Why it matters
In LLM frameworks, weak designs around Query vs Chat Engines typically surface as framework lock-in, opaque magic, and untested compositions. Investing here improves reliability, debuggability, and the ability to change models later.

### How it works
List options, define criteria (quality, cost, latency, ops, lock-in), score with evidence, document the decision.

### Intuition
Explain Query vs Chat Engines as an orchestration layer—useful only when it clarifies ownership of steps. If a new engineer cannot tell what is trusted input, what is allowed action, and what 'done' means, the design is still fuzzy.

### Pitfalls
- Treating Query vs Chat Engines as a one-time playground experiment instead of a versioned artifact
- No success criteria or eval set for Query vs Chat Engines
- Ignoring cost/latency tradeoffs while chasing marginal quality
- Missing adversarial cases typical of LLM frameworks: framework lock-in, opaque magic, and untested compositions

### When to use
Use Query vs Chat Engines when your product path depends on this concern in LLM frameworks. Prefer the simplest design that meets quality, latency, and safety budgets—and prove it with fixtures.


In [ ]:
# Demo: make "Query vs Chat Engines" concrete as a checkable contract
from dataclasses import dataclass, field, asdict
import json

@dataclass
class ConceptContract:
    name: str = "Query vs Chat Engines"
    notebook: str = "03-llamaindex"
    must_have: list = field(default_factory=lambda: [
        "clear inputs/outputs",
        "failure behavior defined",
        "eval fixtures exist",
    ])
    risks: list = field(default_factory=lambda: [
        "silent quality drift",
        "unbounded cost/latency",
    ])

    def health(self) -> dict:
        return {
            "concept": self.name,
            "checks": len(self.must_have),
            "risks": len(self.risks),
            "ready_for_design_review": len(self.must_have) >= 3,
        }

contract_5 = ConceptContract()
print(json.dumps({"contract": asdict(contract_5), "health": contract_5.health()}, indent=2))


In [ ]:
# Demo: before/after quality rubric for "Query vs Chat Engines"
def score_artifact(artifact: dict, rubric: list[str]) -> dict:
    missing = [r for r in rubric if not artifact.get(r)]
    return {"score": round(1 - len(missing)/max(1,len(rubric)), 2), "missing": missing}

rubric = ["definition", "example", "failure_mode", "metric"]
weak = {"definition": "Query vs Chat Engines"}
strong = {"definition": "Query vs Chat Engines", "example": "worked example", "failure_mode": "empty input", "metric": "exact_match"}
print("weak", score_artifact(weak, rubric))
print("strong", score_artifact(strong, rubric))


In [ ]:
# Demo: operational checklist runner for "Query vs Chat Engines"
checks = {
    "has_owner": True,
    "has_eval_set": True,
    "has_token_budget": False,
    "has_alert": False,
}
failed = [k for k, ok in checks.items() if not ok]
print({"topic": "Query vs Chat Engines", "passed": len(checks)-len(failed), "failed": failed})


### Worked scenario — Query vs Chat Engines

**Situation:** A team wants to productionize a feature involving **Query vs Chat Engines**.

**Walkthrough:**
1. Write a one-sentence success metric.
2. Define inputs, outputs, and hard constraints.
3. Implement the smallest demo that can fail loudly.
4. Add one adversarial fixture (empty, hostile, or oversized input).
5. Decide ship/no-ship using the metric—not eloquence.


## Comparison Snapshot

Use this table when reviewing designs in **LlamaIndex**.

| Topic | Do | Don't |
|-------|----|-------|
| Core Concepts | Design carefully; measure; bound cost | Skipping eval / unbounded loops |
| Data Connectors | Design carefully; measure; bound cost | Skipping eval / unbounded loops |
| Indexing & Nodes | Design carefully; measure; bound cost | Skipping eval / unbounded loops |
| Retrieval | Design carefully; measure; bound cost | Skipping eval / unbounded loops |
| RAG Pipeline | Design carefully; measure; bound cost | Skipping eval / unbounded loops |
| Query vs Chat Engines | Design carefully; measure; bound cost | Skipping eval / unbounded loops |


## Glossary / Key Terms

| Term | Meaning |
|------|---------|
| Core Concepts | Key concept covered in this notebook; see its section for definition and pitfalls |
| Data Connectors | Key concept covered in this notebook; see its section for definition and pitfalls |
| Indexing & Nodes | Key concept covered in this notebook; see its section for definition and pitfalls |
| Retrieval | Key concept covered in this notebook; see its section for definition and pitfalls |
| RAG Pipeline | Key concept covered in this notebook; see its section for definition and pitfalls |
| Query vs Chat Engines | Key concept covered in this notebook; see its section for definition and pitfalls |


## Summary & Key Takeaways

- **LlamaIndex** is a production concern: contracts, evals, and guardrails beat vibe-driven prompting.
- Every major topic above includes definition, motivation, mechanism, intuition, pitfalls, and usage guidance—use that checklist in design reviews.
- Prefer small, measurable demos before framework sprawl.
- Bound loops, validate tool args, and keep API keys in environment variables (`YOUR_API_KEY` is a placeholder only).
- Carry forward: connect these ideas to the next notebooks in **09-llm-frameworks**.


## Try It Yourself

1. Implement a failing test/fixture for **Core Concepts**, then fix your demo until it passes.
2. Implement a failing test/fixture for **Data Connectors**, then fix your demo until it passes.
3. Implement a failing test/fixture for **Indexing & Nodes**, then fix your demo until it passes.
4. Implement a failing test/fixture for **Retrieval**, then fix your demo until it passes.
5. Implement a failing test/fixture for **RAG Pipeline**, then fix your demo until it passes.
6. Estimate token cost for your prompt/tool trace at 1k and 100k requests/day.
7. Write a 5-row comparison of two design alternatives from this notebook; pick one with explicit criteria.
8. Red-team your solution with empty input, hostile input, and a tool/API timeout.
